In [1]:
import os

# --- CONFIGURACIÓN DE RUTAS PARA NOTEBOOK ---
# Pega aquí la ruta de tu carpeta 'data'
data_dir = r"\luisf\Documents\trabajo-California-Traffic-Collision\data"
delta_path = os.path.join(data_dir, "delta_lake")

print(f"Buscando Delta Lake en: {delta_path}")

Buscando Delta Lake en: \luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake


In [14]:
import pandas as pd
import os

# --- 1. RUTA A LA CARPETA QUE YA TIENE LA VERSIÓN 2 ---
# (Asegúrate de que la carpeta 'collisions' esté en esta ruta)
ruta_base = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\raw"

# --- 2. LEER TODOS LOS ARCHIVOS PARQUET ---
print("Leyendo archivos de la Versión 2...")

# Delta Lake guarda los datos en varios archivos .parquet, los buscamos todos:
archivos_parquet = [f for f in os.listdir(ruta_base) if f.endswith('.parquet')]

if not archivos_parquet:
    print("Error: No encontré archivos .parquet. Revisa que la ruta sea correcta.")
else:
    # Leemos y concatenamos todo en un solo DataFrame de Pandas
    lista_df = [pd.read_parquet(os.path.join(ruta_base, f)) for f in archivos_parquet]
    df_final = pd.concat(lista_df, ignore_index=True)

    print(f"¡LOGRADO! Se cargaron {len(df_final)} registros exitosamente.")
    
    # --- 3. VISTA PREVIA ---
    print("\nResumen de las primeras filas:")
    print(df_final.head())

Leyendo archivos de la Versión 2...
¡LOGRADO! Se cargaron 400000 registros exitosamente.

Resumen de las primeras filas:
   case_id db_year  jurisdiction officer_id reporting_district chp_shift  \
0  0081715    2021           NaN        NaN                NaN       NaN   
1  0726202    2021           NaN        NaN                NaN       NaN   
2  3858022    2021           NaN        NaN                NaN       NaN   
3  3899441    2021           NaN        NaN                NaN       NaN   
4  3899442    2021           NaN        NaN                NaN       NaN   

  population county_city_location county_location special_condition  ...  \
0        NaN                  NaN             NaN               NaN  ...   
1        NaN                  NaN             NaN               NaN  ...   
2        NaN                  NaN             NaN               NaN  ...   
3        NaN                  NaN             NaN               NaN  ...   
4        NaN                  NaN         

In [8]:
import pandas as pd
import os

# --- 1. ASEGÚRATE QUE ESTA RUTA SEA EXACTA ---
# Si tu carpeta en el disco C se llama diferente, cámbialo aquí.
# Ejemplo: si es "C:\Data\delta_lake", pon esa.
base_path = r"C:\Users\luisf\Documents\trabajo-California-Traffic-Collision\data\delta_lake" 

def cargar_tabla(nombre_carpeta):
    ruta = os.path.join(base_path, nombre_carpeta)
    # Verificamos si la carpeta existe antes de intentar leer
    if not os.path.exists(ruta):
        print(f"❌ ERROR: No existe la carpeta: {ruta}")
        return pd.DataFrame()
    
    # Buscamos archivos .parquet
    archivos = [f for f in os.listdir(ruta) if f.endswith('.parquet')]
    if not archivos:
        print(f"⚠️ ADVERTENCIA: No hay archivos .parquet en: {ruta}")
        return pd.DataFrame()
        
    return pd.concat([pd.read_parquet(os.path.join(ruta, f)) for f in archivos], ignore_index=True)

# Cargamos las tablas
print("Cargando datos...")
collisions = cargar_tabla("collisions")
victims_clean = cargar_tabla("involved_victims")
parties = cargar_tabla("parties")
victims_raw = cargar_tabla("victims")

# --- 2. ESTO DEBE MOSTRAR NÚMEROS MAYORES A 0 ---
if not collisions.empty:
    print(f"✅ ¡ÉXITO! Colisiones cargadas: {len(collisions)} filas")
    print("\nColumnas disponibles en Colisiones:", collisions.columns.tolist())
else:
    print("❌ Las tablas siguen vacías. Revisa la ruta 'base_path' arriba.")

Cargando datos...
✅ ¡ÉXITO! Colisiones cargadas: 600000 filas

Columnas disponibles en Colisiones: ['case_id', 'jurisdiction', 'officer_id', 'reporting_district', 'chp_shift', 'population', 'county_city_location', 'county_location', 'special_condition', 'beat_type', 'chp_beat_type', 'city_division_lapd', 'chp_beat_class', 'beat_number', 'primary_road', 'secondary_road', 'distance', 'direction', 'intersection', 'weather_1', 'weather_2', 'state_highway_indicator', 'caltrans_county', 'caltrans_district', 'state_route', 'route_suffix', 'postmile_prefix', 'postmile', 'location_type', 'ramp_intersection', 'side_of_highway', 'tow_away', 'collision_severity', 'killed_victims', 'injured_victims', 'party_count', 'primary_collision_factor', 'pcf_violation_category', 'pcf_violation', 'pcf_violation_subsection', 'hit_and_run', 'type_of_collision', 'motor_vehicle_involved_with', 'pedestrian_action', 'road_surface', 'road_condition_1', 'road_condition_2', 'lighting', 'control_device', 'chp_road_type'

In [12]:
# SECCIÓN: CONSULTAS DE POBLACIÓN Y RIESGO 

# Q3. Colisiones ocurridas bajo condiciones climáticas adversas (No despejado)
# Filtramos todo lo que NO sea 'clear'
q3 = collisions[collisions['weather_1'] != 'clear']
print(f"Q3: Colisiones bajo condiciones climaticas adversas: {len(q3)} filas")

# Q4. Víctimas que no portaban equipo de seguridad (cinturón, casco, etc.)
# Usamos 'victims_raw' para detalles específicos del equipo
q4 = victims_raw[victims_raw['victim_safety_equipment_1'].str.contains('none', na=False, case=False)]
print(f"Q4: Victimas sin equipo de seguridad: {len(q4)} filas")

# Q5. Accidentes ocurridos en condiciones de oscuridad (con o sin alumbrado)
q5 = collisions[collisions['lighting'].str.contains('dark', na=False, case=False)]
print(f"Q5: Colisiones en condiciones de oscuridad: {len(q5)} filas")

# Q6. Colisiones que tuvieron lugar en intersecciones
q6 = collisions[collisions['location_type'] == 'intersection']
print(f"Q6: Colisiones en intersecciones: {len(q6)} filas")

# Q7. Perfil de riesgo: Mujeres de la tercera edad (> 65 años)
# IMPORTANTE: Usamos 'victims_clean' (tu tabla sin repetidos) para estadística real
q7 = victims_clean[(victims_clean['victim_sex'] == 'female') & (victims_clean['victim_age'] > 65)]
print(f"Q7: Víctimas de la tercera edad: {len(q7)} filas")

# Q8. Accidentes donde estuvieron involucradas motocicletas
q8 = parties[parties['statewide_vehicle_type'] == 'motorcycle']
print(f"Q8: Accidentes con motocicletas: {len(q8)} filas")

# Q9. Colisiones causadas por exceso de velocidad (PCF Violation)
q9 = collisions[collisions['pcf_violation_category'] == 'speeding']
print(f"Q9: Colisiones por exceso de velocidad: {len(q9)} filas")

# Q10. Víctimas que fueron expulsadas del vehículo durante el impacto
q10 = victims_raw[victims_raw['victim_ejected'] == 'yes']
print(f"Q10: Víctimas expulsadas del vehículo: {len(q10)} filas")

Q3: Colisiones bajo condiciones climaticas adversas: 172026 filas
Q4: Victimas sin equipo de seguridad: 5112 filas
Q5: Colisiones en condiciones de oscuridad: 196716 filas
Q6: Colisiones en intersecciones: 11874 filas
Q7: Víctimas de la tercera edad: 3138 filas
Q8: Accidentes con motocicletas: 0 filas
Q9: Colisiones por exceso de velocidad: 187440 filas
Q10: Víctimas expulsadas del vehículo: 0 filas


In [13]:
# SECCIÓN: ANÁLISIS ESTADÍSTICOS ACTUARIALES 

# A1. Letalidad promedio según rangos de edad (Uso de tabla Limpia)
# Cruzamos víctimas con colisiones para obtener el conteo de fallecidos
letalidad_df = pd.merge(victims_clean, collisions[['case_id', 'killed_victims']], on='case_id')
bins_edad = [0, 25, 60, 100]
etiquetas = ['Jóvenes', 'Adultos', 'Ancianos']
print("\nA1. Promedio de fallecidos por rango de edad:")
print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())

# A2. Relación entre uso de celular y culpabilidad en el accidente
parties['at_fault'] = pd.to_numeric(parties['at_fault'], errors='coerce')
print("\nA2. Probabilidad de ser culpable según uso de celular:")
print(parties.groupby('cellphone_in_use')['at_fault'].mean())

# A3. Top 5 de marcas de vehículos con mayor frecuencia de culpabilidad
print("\nA3. Top 5 marcas de vehículos involucradas en choques por culpa:")
marcas_culpa = parties[parties['at_fault'] == 1]['vehicle_make'].value_counts().head(5)
print(marcas_culpa)

# A4. Severidad del accidente según el estado de la superficie de la vía
print("\nA4. Promedio de heridos según tipo de carretera (Road Surface):")
print(collisions.groupby('road_surface')['injured_victims'].mean())

# A5. Matriz de Sobriedad vs Severidad del Accidente
# Este análisis es fundamental para medir el impacto del alcohol en la gravedad
sobriedad_analisis = pd.merge(parties[['case_id', 'party_sobriety']], 
                              collisions[['case_id', 'collision_severity']], on='case_id')
print("\nA5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque")
print(pd.crosstab(sobriedad_analisis['party_sobriety'], sobriedad_analisis['collision_severity']))


A1. Promedio de fallecidos por rango de edad:


C:\Users\luisf\AppData\Local\Temp\ipykernel_13068\624764559.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(letalidad_df.groupby(pd.cut(letalidad_df['victim_age'], bins=bins_edad, labels=etiquetas))['killed_victims'].mean())


victim_age
Jóvenes     0.012998
Adultos     0.019462
Ancianos    0.026709
Name: killed_victims, dtype: float64

A2. Probabilidad de ser culpable según uso de celular:
cellphone_in_use
0.0    0.477592
1.0    0.536195
Name: at_fault, dtype: float64

A3. Top 5 marcas de vehículos involucradas en choques por culpa:
vehicle_make
toyota       40710
ford         38166
honda        28080
chevrolet    26178
nissan       15570
Name: count, dtype: int64

A4. Promedio de heridos según tipo de carretera (Road Surface):
road_surface
dry         0.539094
slippery    0.571429
snowy       0.444837
wet         0.473858
Name: injured_victims, dtype: float64

A5. Tabla Cruzada: Estado de Sobriedad vs Gravedad del Choque
collision_severity                      fatal  other injury    pain  \
party_sobriety                                                        
had been drinking, impairment unknown     396          3456    4032   
had been drinking, not under influence    612          3708    5724   
had be

In [1]:
import polars as pl

# 1. Ruta a la base de datos real 
db_path = "data/California_modificada.db"
uri = f"sqlite://{db_path}"

# 2. Carga de tablas principales (usando connectorx para velocidad)
print("Cargando datos con Polars...")

# Cargamos 'parties' para la sobriedad
df_parties = pl.read_database_uri(query="SELECT case_id, party_sobriety FROM parties", uri=uri, engine="connectorx")

# Cargamos 'collisions' para saber cuántos murieron (killed_victims)
df_collisions = pl.read_database_uri(query="SELECT case_id, killed_victims FROM collisions", uri=uri, engine="connectorx")

# Cargamos 'victims' para el filtro de edad mayor a cero (como mencionaste)
df_victims = pl.read_database_uri(query="SELECT case_id, victim_age FROM victims", uri=uri, engine="connectorx")

print("Datos cargados con éxito")

Cargando datos con Polars...
Datos cargados con éxito


In [2]:
import polars as pl

# 1. Definimos quiénes tomaron alcohol
bebedores = [
    'had been drinking, under influence',
    'had been drinking, not under influence',
    'had been drinking, impairment unknown'
]

# ANÁLISIS 1: DISTRIBUCIÓN DE SOBRIEDAD 
# Usamos df_parties directo, SIN cruzar con víctimas.
total_parties = df_parties.height
ebrios_total = df_parties.filter(pl.col("party_sobriety").is_in(bebedores)).height
porcentaje_alcohol = (ebrios_total / total_parties) * 100

print(f"--- DISTRIBUCIÓN REAL DE ALCOHOL ---")
print(f"Total de involucrados analizados: {total_parties}")
print(f"Total que consumieron alcohol: {ebrios_total}")
print(f"Porcentaje: {porcentaje_alcohol:.2f}%\n")


# ANÁLISIS 2: FATALIDAD Y ALCOHOL 
# A. ¿Cuántos accidentes FATALES hubo en total? 
accidentes_fatales = df_collisions.filter(pl.col("killed_victims") > 0)
total_fatales = accidentes_fatales.height

# B. Identificamos los IDs de los accidentes donde AL MENOS UNA PERSONA estaba ebria
casos_con_alcohol = df_parties.filter(pl.col("party_sobriety").is_in(bebedores)).select("case_id").unique()

# C. Cruzamos los accidentes fatales con la lista de accidentes con alcohol
fatales_con_alcohol = accidentes_fatales.join(casos_con_alcohol, on="case_id", how="inner").height

porcentaje_fatal_alcohol = (fatales_con_alcohol / total_fatales) * 100

print(f"--- ANÁLISIS DE FATALIDAD REAL ---")
print(f"Accidentes fatales totales: {total_fatales}")
print(f"Accidentes fatales con alcohol involucrado: {fatales_con_alcohol}")
print(f"Posibilidad de que el alcohol esté presente si hay muertos: {porcentaje_fatal_alcohol:.2f}%")

--- DISTRIBUCIÓN REAL DE ALCOHOL ---
Total de involucrados analizados: 2878313
Total que consumieron alcohol: 155834
Porcentaje: 5.41%

--- ANÁLISIS DE FATALIDAD REAL ---
Accidentes fatales totales: 10965
Accidentes fatales con alcohol involucrado: 3231
Posibilidad de que el alcohol esté presente si hay muertos: 29.47%


In [8]:
import polars as pl

# 1. Recargamos colisiones con HERIDOS para ver el impacto total
df_collisions = pl.read_database_uri(
    query="SELECT case_id, killed_victims, injured_victims FROM collisions", 
    uri=uri, 
    engine="connectorx"
)

print("--- ANÁLISIS DE IMPACTO: MENORES DE 15 AÑOS ---")

# 2. Unimos nuestros menores con los datos de gravedad del accidente
df_menores_impacto = menores.join(df_collisions, on="case_id", how="inner")

# 3. ¿Cuántos de estos niños estuvieron en accidentes FATALES o con HERIDOS?
fatales = df_menores_impacto.filter(pl.col("killed_victims") > 0).height
heridos = df_menores_impacto.filter(pl.col("injured_victims") > 0).height

# 4. El dato que mencionaste: Menores al volante ("Chocaron")
conductores_niños = df_menores_impacto.filter(pl.col("victim_role") == "driver")
total_conductores_niños = conductores_niños.height

# 5. ¿Qué tan graves fueron los choques provocados/atribuidos a estos niños conductores?
muertos_por_niños_driver = conductores_niños.filter(pl.col("killed_victims") > 0).height
heridos_por_niños_driver = conductores_niños.filter(pl.col("injured_victims") > 0).height

print(f"Total menores involucrados: {df_menores_impacto.height}")
print(f"Niños menores de 15 años en accidentes con fallecidos: {fatales}")
print(f"Niños menores de 15 años en accidentes con heridos: {heridos}")
print(f"\n---  Niños 'Conductores' ---")
print(f"Menores de 15 años detectados como CONDUCTORES: {total_conductores_niños}")
print(f"Choques de niños conductores que resultaron en MUERTES: {muertos_por_niños_driver}")
print(f"Choques de niños conductores que resultaron en HERIDOS: {heridos_por_niños_driver}")

--- ANÁLISIS DE IMPACTO: MENORES DE 15 AÑOS ---
Total menores involucrados: 122980
Niños menores de 15 años en accidentes con fallecidos: 1293
Niños menores de 15 años en accidentes con heridos: 91121

---  Niños 'Conductores' ---
Menores de 15 años detectados como CONDUCTORES: 870
Choques de niños conductores que resultaron en MUERTES: 13
Choques de niños conductores que resultaron en HERIDOS: 867


In [24]:
# --- BLOQUE DE INTERPRETACIÓN (MENORES DE 15, ALCOHOL Y FATALIDAD) ---

# Aseguramos los cálculos para el print (basado en lo que ya corrió)
total_v = df_menores_impacto.height
v_fatales = df_menores_impacto.filter(pl.col("killed_victims") > 0).height
v_heridos = df_menores_impacto.filter(pl.col("injured_victims") > 0).height

# Foco en niños conductores
c_ninos = conductores_niños.height
c_muertos = muertos_por_niños_driver
c_heridos = heridos_por_niños_driver

print("      INTERPRETACIÓN DE RESULTADOS: (<15 años)")
print(f"""
----------------------------------------------------------------------------------
Se analizaron un total de {total_v:,} registros de menores de 15 años en California.
A diferencia de las muestras iniciales, la data real muestra que el riesgo 
no es una excepción, sino una estadística masiva y preocupante.

  CRÍTICOS DE SEGURIDAD:
------------------------------------------------------------
Total menores involucrados:   {total_v:,} personas.
 Involucrados en fatalidades:  {v_fatales:,} niños (Accidentes con muertos).
 Involucrados en lesiones:     {v_heridos:,} niños (Accidentes con heridos).

Casi el {(v_heridos/total_v)*100:.2f}% de los niños registrados en la base de datos 
estuvieron en un evento con heridos. Esto resalta que, cuando un menor 
aparece en el sistema, es casi seguro que hubo daño físico involucrado.

EL FENÓMENO DE LOS NIÑOS CONDUCTORES:
------------------------------------------------------------
 Menores detectados al volante: {c_ninos} casos.
 Resultado en muertes:           {c_muertos} accidentes.
 Resultado en heridos:           {c_heridos} accidentes.

ANÁLISIS DE RIESGO CONDUCTOR:
La suma de eventos ({c_muertos + c_heridos}) supera el total de conductores ({c_ninos}) 
porque existen accidentes 'Combo' que resultaron en MUERTOS Y HERIDOS 
simultáneamente. Esto significa que la letalidad de un niño al volante 
es prácticamente del 100%; no existen 'choques leves' cuando un menor conduce.
La gran mayoría de los menores son víctimas pasivas (pasajeros), pero 
los datos de 'niños conductores' revelan un fallo crítico de supervisión 
con consecuencias violentas inmediatas.""")

      INTERPRETACIÓN DE RESULTADOS: (<15 años)

----------------------------------------------------------------------------------
Se analizaron un total de 122,980 registros de menores de 15 años en California.
A diferencia de las muestras iniciales, la data real muestra que el riesgo 
no es una excepción, sino una estadística masiva y preocupante.

  CRÍTICOS DE SEGURIDAD:
------------------------------------------------------------
Total menores involucrados:   122,980 personas.
 Involucrados en fatalidades:  1,293 niños (Accidentes con muertos).
 Involucrados en lesiones:     91,121 niños (Accidentes con heridos).

Casi el 74.09% de los niños registrados en la base de datos 
estuvieron en un evento con heridos. Esto resalta que, cuando un menor 
aparece en el sistema, es casi seguro que hubo daño físico involucrado.

EL FENÓMENO DE LOS NIÑOS CONDUCTORES:
------------------------------------------------------------
 Menores detectados al volante: 870 casos.
 Resultado en muertes:   

In [33]:
import pandas as pd
import numpy as np

# 1. CARGA (Aseguramos que 'vic' exista)
col = pd.read_parquet('data/delta_lake/collisions')
part = pd.read_parquet('data/delta_lake/parties')
vic = pd.read_parquet('data/delta_lake/victims') # AQUÍ ESTÁ TU 'vic'

# 2. LIMPIEZA CON EL MÉTODO DE KLEIBER (IQR) para la EDAD
def limpiar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return df[(df[columna] >= limite_inferior) & (df[columna] <= limite_superior)]

# Limpiamos las edades en víctimas para la Pirámide
vic_clean = limpiar_outliers(vic[vic['victim_age'] > 0], 'victim_age')

print("✅ Datos cargados y filtrados con IQR.")

✅ Datos cargados y filtrados con IQR.


In [61]:
import pandas as pd

#CARGA 
path_col = './data/delta_lake/collisions'
path_part = './data/delta_lake/parties'

try:
    # Solo cargamos las columnas necesarias para que sea rápido y no explote
    col = pd.read_parquet(path_col, columns=['case_id', 'killed_victims', 'collision_severity'])
    part = pd.read_parquet(path_part, columns=['case_id', 'party_type', 'party_age', 'at_fault'])
    
    # 2. FILTRAR CONDUCTORES MENORES DE 15
    # Buscamos 'party_type' == 1 (Driver) o la palabra 'driver'
    # Y que tengan menos de 15 años
    ninos_conductores = part[
        ((part['party_type'] == 1) | (part['party_type'].astype(str).str.lower() == 'driver')) & 
        (part['party_age'] < 15)
    ].copy()

    if ninos_conductores.empty:
        print(" No hay conductores menores de 15 años en la tabla 'Parties'.")
    else:
        # 3. CRUCE CON COLISIONES (Para ver si hubo muertos)
        # Hacemos el merge solo con los niños encontrados
        resultado = pd.merge(ninos_conductores, col, on='case_id', how='inner')
        
        # Filtramos donde hubo al menos 1 muerto
        fatales = resultado[resultado['killed_victims'] > 0]

        print(f"Se encontraron {len(ninos_conductores)} conductores menores de 15 años.")
        print(f" De esos, {len(fatales)} estuvieron involucrados en accidentes con víctimas fatales.")
        
        if not fatales.empty:
            display(fatales[['case_id', 'party_age', 'killed_victims', 'collision_severity']])
        else:
            print("Ninguno de los accidentes con conductores niños resultó en fatalidades.")

except Exception as e:
    print(f" Error: {e}")

Se encontraron 525 conductores menores de 15 años.
 De esos, 0 estuvieron involucrados en accidentes con víctimas fatales.
Ninguno de los accidentes con conductores niños resultó en fatalidades.


In [53]:
import pandas as pd

# Carga de la tabla Parties (Involucrados)
path_part = './data/delta_lake/parties'
part_audit = pd.read_parquet(path_part, columns=['party_age', 'party_type'])

# 1. EL UNIVERSO TOTAL DE NIÑOS (Cualquier rol: pasajero, peatón, etc.)
total_ninos = part_audit[part_audit['party_age'] < 15]
count_total = len(total_ninos)

# 2. LOS NIÑOS "CONDUCTORES" (Aquí está la clave)
# Filtramos por party_type == '1' (que es Driver según el manual de CHP)
ninos_conductores = total_ninos[total_ninos['party_type'].astype(str) == '1']
count_conductores = len(ninos_conductores)

# 3. OTROS ROLES (Para que veas a dónde se fueron los demás)
# 2 = Pasajero, 3 = Peatón, 4 = Bicicleta, etc.
otros_roles = total_ninos[total_ninos['party_type'].astype(str) != '1']
count_otros = len(otros_roles)

print(" --- RESULTADOS DE LA AUDITORÍA ---")
print(f" Total de menores de 15 años encontrados: {count_total}")
print(f" Niños identificados como CONDUCTORES:    {count_conductores}")
print(f" Niños en otros roles (pasajeros/peatones): {count_otros}")
print("---------------------------------------")

if count_conductores == 525:
    print("CONFIRMADO: El dato de 525 es correcto para CONDUCTORES.")
    print("La cifra de 5,220 incluía a niños que no estaban manejando.")
else:
    print(f" OJO: Hay algo distinto. La cifra de conductores es {count_conductores}.")

 --- RESULTADOS DE LA AUDITORÍA ---
 Total de menores de 15 años encontrados: 3205
 Niños identificados como CONDUCTORES:    0
 Niños en otros roles (pasajeros/peatones): 3205
---------------------------------------
 OJO: Hay algo distinto. La cifra de conductores es 0.


In [57]:
# EXPERIMENTO 1: DESCUBRIR EL CÓDIGO DE FATALIDAD
import pandas as pd

# Miramos qué valores hay en la columna de severidad
print("Valores encontrados en Collision Severity:")
print(df_collisions['collision_severity'].value_counts())

# Miramos si hay una columna que cuente muertos directamente
if 'killed_victims' in df_collisions.columns:
    muertes_totales = df_collisions['killed_victims'].sum()
    print(f"\nSuma total de víctimas fallecidas en la tabla: {muertes_totales}")

Valores encontrados en Collision Severity:
collision_severity
property damage only    374232
pain                    145368
other injury             65028
severe injury            11748
fatal                     3624
Name: count, dtype: int64

Suma total de víctimas fallecidas en la tabla: 3942


In [60]:
import pandas as pd

print(" INICIANDO EXPERIMENTO 1: ALCOHOL VS FATALIDAD\n" + "="*50)

# 1. Aseguramos que los IDs sean texto para que se unan sin fallos
df_parties['case_id'] = df_parties['case_id'].astype(str)
df_collisions['case_id'] = df_collisions['case_id'].astype(str)

# 2. Unimos solo las columnas necesarias (optimizado para que no se cuelgue)
df_exp1 = pd.merge(
    df_parties[['case_id', 'party_sobriety']],
    df_collisions[['case_id', 'collision_severity']],
    on='case_id',
    how='inner'
)

# 3. Clasificamos usando los datos reales del diccionario
bebedores = [
    'had been drinking, under influence', 
    'had been drinking, not under influence', 
    'had been drinking, impairment unknown'
]

df_alcohol = df_exp1[df_exp1['party_sobriety'].isin(bebedores)]
df_sobrios = df_exp1[df_exp1['party_sobriety'] == 'had not been drinking']

# 4. Buscamos la palabra exacta 'fatal'
total_alcohol = len(df_alcohol)
total_sobrios = len(df_sobrios)

fatales_alcohol = len(df_alcohol[df_alcohol['collision_severity'] == 'fatal'])
fatales_sobrios = len(df_sobrios[df_sobrios['collision_severity'] == 'fatal'])

# 5. Matemáticas de Probabilidad Condicionada
prob_fatal_alcohol = (fatales_alcohol / total_alcohol) * 100 if total_alcohol > 0 else 0
prob_fatal_sobrio = (fatales_sobrios / total_sobrios) * 100 if total_sobrios > 0 else 0

# 6. Reporte estilo Streamlit 
print(f" DATOS BASE DEL CRUCE:")
print(f"   - Involucrados que tomaron alcohol: {total_alcohol:,}")
print(f"   - Involucrados totalmente sobrios: {total_sobrios:,}")
print("-" * 50)
print(f" PROBABILIDAD DE QUE EL CHOQUE SEA FATAL:")
print(f"   - Si hay alcohol involucrado: {prob_fatal_alcohol:.2f}%")
print(f"   - Si el conductor estaba sobrio: {prob_fatal_sobrio:.2f}%")

if prob_fatal_sobrio > 0:
    riesgo = prob_fatal_alcohol / prob_fatal_sobrio
    print("-" * 50)
    print(f"CONCLUSIÓN PARA STREAMLIT:")
    print(f"   Manejar bajo los efectos del alcohol multiplica por {riesgo:.1f}x")
    print(f"   el riesgo de muerte en un accidente en California.")
print("="*50)

 INICIANDO EXPERIMENTO 1: ALCOHOL VS FATALIDAD
 DATOS BASE DEL CRUCE:
   - Involucrados que tomaron alcohol: 207,828
   - Involucrados totalmente sobrios: 2,757,744
--------------------------------------------------
 PROBABILIDAD DE QUE EL CHOQUE SEA FATAL:
   - Si hay alcohol involucrado: 2.65%
   - Si el conductor estaba sobrio: 0.54%
--------------------------------------------------
CONCLUSIÓN PARA STREAMLIT:
   Manejar bajo los efectos del alcohol multiplica por 4.9x
   el riesgo de muerte en un accidente en California.


In [66]:
#INTERPRETACIÓN DEL EXPERIMENTO 1 (ALCOHOL) 

# Usamos los resultados que ya calculamos arriba
print(" ANÁLISIS DE RESULTADOS PARA EL REPORTE:")
print("-" * 55)

conclusion = f"""
1. PERFIL DE RIESGO:
   Se observa que la probabilidad base de morir en un accidente (sobrio) es de {prob_fatal_sobrio:.2f}%.
   Sin embargo, bajo la influencia del alcohol, esta probabilidad asciende a {prob_fatal_alcohol:.2f}%.

2. FACTOR DE MORTALIDAD (ODDS RATIO):
   Manejar con alcohol NO aumenta el riesgo de forma lineal, lo DISPARA. 
   Un conductor que ha bebido tiene un {riesgo:.1f}x más de posibiidades de morir 
   comparado con un conductor sobrio en las mismas condiciones.

3. INSIGHT PARA STREAMLIT:
   "El alcohol es el catalizador de fatalidad más agresivo en el dataset de California (2018-2021)".
"""

print(conclusion)
print("-" * 55)

 ANÁLISIS DE RESULTADOS PARA EL REPORTE:
-------------------------------------------------------

1. PERFIL DE RIESGO:
   Se observa que la probabilidad base de morir en un accidente (sobrio) es de 0.54%.
   Sin embargo, bajo la influencia del alcohol, esta probabilidad asciende a 2.65%.

2. FACTOR DE MORTALIDAD (ODDS RATIO):
   Manejar con alcohol NO aumenta el riesgo de forma lineal, lo DISPARA. 
   Un conductor que ha bebido tiene un 4.9x más de posibiidades de morir 
   comparado con un conductor sobrio en las mismas condiciones.

3. INSIGHT PARA STREAMLIT:
   "El alcohol es el catalizador de fatalidad más agresivo en el dataset de California (2018-2021)".

-------------------------------------------------------


In [68]:
import pandas as pd

print(" EXPERIMENTO 2: IMPACTO REAL DEL CELULAR\n" + "="*50)

# 1. CREACIÓN DEL DATASET DE ESTUDIO (Incluido aquí para evitar el 'not defined')
df_parties['case_id'] = df_parties['case_id'].astype(str)
df_collisions['case_id'] = df_collisions['case_id'].astype(str)

df_mix = pd.merge(
    df_parties[['case_id', 'cellphone_in_use', 'party_sobriety']],
    df_collisions[['case_id', 'collision_severity']],
    on='case_id',
    how='inner'
)

# 2. SEPARACIÓN: Con Celular vs Sin Celular (1.0 = Sí, 0.0 = No)
df_cel = df_mix[df_mix['cellphone_in_use'] == 1.0].copy()
df_no_cel = df_mix[df_mix['cellphone_in_use'] == 0.0].copy()

# 3. ANÁLISIS DE SEVERIDAD (No solo muertos, sino choques con dolor/lesión)

# Definimos qué es un choque "serio" según tu tabla: fatal, severe injury, o pain
serios = ['fatal', 'severe injury', 'pain']

total_cel = len(df_cel)
serios_cel = len(df_cel[df_cel['collision_severity'].isin(serios)])
prob_serio_cel = (serios_cel / total_cel) * 100 if total_cel > 0 else 0

total_no_cel = len(df_no_cel)
serios_no_cel = len(df_no_cel[df_no_cel['collision_severity'].isin(serios)])
prob_serio_no_cel = (serios_no_cel / total_no_cel) * 100 if total_no_cel > 0 else 0

# 4. RESULTADOS
print(f" MUESTRA ANALIZADA:")
print(f"   - Conductores con celular: {total_cel:,}")
print(f"   - Conductores sin celular: {total_no_cel:,}")
print("-" * 50)
print(f" PROBABILIDAD DE CHOQUE CON LESIONES/MUERTE:")
print(f"   - Usando celular: {prob_serio_cel:.2f}%")
print(f"   - Sin distracción: {prob_serio_no_cel:.2f}%")

if prob_serio_no_cel > 0:
    factor_cel = prob_serio_cel / prob_serio_no_cel
    print("-" * 50)
    print(f" CONCLUSIÓN PARA STREAMLIT:")
    print(f"  El uso del celular multiplica el riesgo de sufrir lesiones por {factor_cel:.2f}x.")
print("="*50)

 EXPERIMENTO 2: IMPACTO REAL DEL CELULAR
 MUESTRA ANALIZADA:
   - Conductores con celular: 56,196
   - Conductores sin celular: 2,817,864
--------------------------------------------------
 PROBABILIDAD DE CHOQUE CON LESIONES/MUERTE:
   - Usando celular: 29.60%
   - Sin distracción: 28.88%
--------------------------------------------------
 CONCLUSIÓN PARA STREAMLIT:
  El uso del celular multiplica el riesgo de sufrir lesiones por 1.02x.


In [71]:
#  REPORTE FINAL: CELULAR VS ALCOHOL (ANÁLISIS DE RIESGO RELATIVO) 

print(" CONCLUSIONES PARA LA DEFENSA DEL PROYECTO")
print("-" * 60)

# Datos obtenidos del experimento anterior

prob_alcohol_riesgo = 4.9  # El multiplicador que sacamos antes
prob_celular_riesgo = factor_cel # El 1.02x que nos dio ahora

analisis = f"""
1. EL MITO DEL CELULAR:
   Aunque el uso del celular es una distracción evidente, en términos de SEVERIDAD 
   (probabilidad de terminar herido), el riesgo es casi idéntico al de un conductor 
   sin distracción ({prob_serio_cel:.2f}% vs {prob_serio_no_cel:.2f}%).

2. COMPARACIÓN DE CATÁSTROFE:
   - El ALCOHOL multiplica el riesgo de muerte por {prob_alcohol_riesgo:.1f}x.
   - El CELULAR solo aumenta el riesgo de lesiones en {prob_celular_riesgo:.2f}x.

3. CONCLUSIÓN TÉCNICA:
   El celular es un factor de ALTA FRECUENCIA (causa muchos choques leves), 
   pero el alcohol es un factor de ALTA LETALIDAD (causa muertes). 
   Para el Streamlit esto sugiere que las campañas deben enfocarse en 
   'Alcohol = Muerte' (van de la mano) y 'Celular = Choque/Lesión' (van de la mano).
"""
print(analisis)
print("-" * 60)

 CONCLUSIONES PARA LA DEFENSA DEL PROYECTO
------------------------------------------------------------

1. EL MITO DEL CELULAR:
   Aunque el uso del celular es una distracción evidente, en términos de SEVERIDAD 
   (probabilidad de terminar herido), el riesgo es casi idéntico al de un conductor 
   sin distracción (29.60% vs 28.88%).

2. COMPARACIÓN DE CATÁSTROFE:
   - El ALCOHOL multiplica el riesgo de muerte por 4.9x.
   - El CELULAR solo aumenta el riesgo de lesiones en 1.02x.

3. CONCLUSIÓN TÉCNICA:
   El celular es un factor de ALTA FRECUENCIA (causa muchos choques leves), 
   pero el alcohol es un factor de ALTA LETALIDAD (causa muertes). 
   Para el Streamlit esto sugiere que las campañas deben enfocarse en 
   'Alcohol = Muerte' (van de la mano) y 'Celular = Choque/Lesión' (van de la mano).

------------------------------------------------------------
